# `date_recorded` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `date_recorded`.

            ## Relationships selected in advance

            - `construction_year` — Together they define waterpoint age at observation.
- `region` — Survey waves may have moved through regions at different times.
- `installer` — Installer activity and recorded construction cohorts may be time-dependent.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'date_recorded'
feature_metadata = {'order': 2, 'name': 'date_recorded', 'audit_type': 'date', 'role': 'candidate', 'disposition': 'replace with elapsed days from a fixed competition-era reference', 'finding': 'Every supplied date parses, but collection timing is concentrated in survey waves.', 'decision': 'Replace the raw date with days_since_recorded from 2015-02-02; do not derive separate calendar components.', 'reference_date': '2015-02-02', 'reference_basis': 'Earliest surviving official DrivenData Pump It Up community post; not asserted as a verified launch timestamp.', 'reference_evidence': 'https://community.drivendata.org/t/about-the-pump-it-up-data-mining-the-water-table-category/63', 'risk': 'Recording time can proxy survey operations and geography rather than waterpoint condition.', 'related': [{'feature': 'construction_year', 'reason': 'Together they define waterpoint age at observation.'}, {'feature': 'region', 'reason': 'Survey waves may have moved through regions at different times.'}, {'feature': 'installer', 'reason': 'Installer activity and recorded construction cohorts may be time-dependent.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for date_recorded.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,construction_year,Together they define waterpoint age at observa...
1,region,Survey waves may have moved through regions at...
2,installer,Installer activity and recorded construction c...


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,date_recorded,construction_year,Spearman correlation,0.1877,59400,356,55,NaN,NaN,Together they define waterpoint age at observa...
1,date_recorded,region,correlation ratio (eta),0.9570,59400,356,21,NaN,NaN,Survey waves may have moved through regions at...
2,date_recorded,installer,correlation ratio (eta),0.7068,59400,356,2146,NaN,NaN,Installer activity and recorded construction c...


In [3]:
recording_year = pd.to_datetime(
    training_features["date_recorded"],
    errors="coerce",
).dt.year
construction_year = pd.to_numeric(
    training_features["construction_year"],
    errors="coerce",
)
age = recording_year.sub(construction_year).where(construction_year.gt(0))
age_check = pd.DataFrame({
    "known construction years": [construction_year.gt(0).sum()],
    "unknown year rows": [construction_year.eq(0).sum()],
    "negative derived ages": [age.lt(0).sum()],
    "median non-negative age": [age.where(age.ge(0)).median()],
    "maximum non-negative age": [age.where(age.ge(0)).max()],
}, index=["date_recorded - construction_year"])
display(age_check)


,known construction years,unknown year rows,negative derived ages,median non-negative age,maximum non-negative age
date_recorded - construction_year,38691,20709,9,13.0,53.0


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `date_recorded`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **replace with elapsed days from a fixed competition-era reference**.
